# European Nucleotide Archive (ENA) — Data Ingestion

**ENA** (European Nucleotide Archive) is the world's most comprehensive public repository of nucleotide sequencing information, maintained by the European Bioinformatics Institute (EMBL-EBI). It archives the full spectrum of sequencing data from raw instrument reads through to assembled and annotated sequences.

## Data hierarchy

```
Study (PRJEB / PRJNA)
 └── Sample (ERS / SRS)          biological sample with metadata
      └── Experiment (ERX / SRX) library preparation details
           └── Run (ERR / SRR)   single sequencing run with FASTQ/BAM files
```

Additional top-level record types include **assemblies** (genome/transcriptome assemblies) and **sequences** (annotated flat-file entries, e.g. EMBL format).

## Key data types

| Record type | Accession prefix | Description |
|---|---|---|
| `study` | PRJEB / PRJNA | Collection of related sequencing experiments |
| `sample` | ERS / SRS | Biological source material with metadata |
| `experiment` | ERX / SRX | Library strategy, instrument, and protocol |
| `run` | ERR / SRR | Actual sequencing run, links to raw data files |
| `assembly` | GCA | Genome or transcriptome assembly |
| `sequence` | ENA accessions | Annotated nucleotide sequences |

**Portal API base:** `https://www.ebi.ac.uk/ena/portal/api/`

**Reference:** Amid et al. (2020). *The European Nucleotide Archive in 2019.* Nucleic Acids Research, 48(D1), D70–D76. https://doi.org/10.1093/nar/gkz1063

# TODO

* [x] **Ingest data**
    * [x] Connect to ENA Portal API and explore available result types and fields
    * [x] Fetch human (taxon 9606) RNA-seq studies, paginate through results, cache to `data/`
    * [x] Parse studies into a Polars DataFrame with correct dtypes
    * [x] Fetch run-level metadata for a study of interest (PRJEB2445)
    * [x] Parse runs into a Polars DataFrame; print shape, dtypes, and head for both
* [ ] **Explore and clean**
    * [ ] Summarise study counts by year, center, and study type
    * [ ] Inspect run-level read/base count distributions
    * [ ] Identify and handle missing values
* [ ] **Analysis**
    * [ ] Characterise instrument model usage trends over time
    * [ ] Compare library strategies and their read depth distributions
    * [ ] Link runs back to sample metadata for biological context
* [ ] **Visualization**
    * [ ] Timeline of RNA-seq study submissions for *H. sapiens*
    * [ ] Read count vs. base count scatter coloured by instrument
    * [ ] Heatmap of library strategy × instrument model usage
* [ ] **Statistical analysis**
    * [ ] Discuss sequencing depth adequacy for differential expression (power analysis)
    * [ ] Explain error models for count data (negative binomial, Poisson) and when each applies
    * [ ] Address multiple hypothesis correction considerations in large-scale RNA-seq analysis

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to ENA Portal API and Explore Available Result Types and Fields

In [ ]:
ENA_BASE = "https://www.ebi.ac.uk/ena/portal/api"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

HUMAN_TAXON = 9606  # NCBI taxon ID for Homo sapiens


def ena_get(endpoint: str, params: dict = None) -> requests.Response:
    """
    Send a GET request to the ENA Portal API with a polite inter-call delay.

    Parameters
    ----------
    endpoint : str
        API path relative to ENA_BASE (e.g. "search" or "returnFields").
    params : dict, optional
        Query parameters to include in the request.

    Returns
    -------
    requests.Response
        Raw response; caller is responsible for parsing (JSON or text).

    Notes
    -----
    A 0.5-second sleep is injected after every call to respect EBI's
    fair-use guidance and avoid triggering rate-limiting.
    """
    url = f"{ENA_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=60)
    resp.raise_for_status()
    time.sleep(0.5)  # polite delay between requests
    return resp


# ── List all available result types ──────────────────────────────────────────
# The /results endpoint enumerates every searchable data type in ENA.
result_types_raw = ena_get("results", {"dataPortal": "ena", "format": "json"}).json()

result_types = pl.DataFrame(result_types_raw)
print(f"Available result types: {len(result_types)}")
print(result_types)

In [ ]:
# ── List available fields for the 'study' result type ────────────────────────
# Each result type exposes a different set of searchable/returnable fields.
# Inspecting them upfront avoids silent 400 errors from invalid field names.
study_fields_raw = ena_get("returnFields", {"result": "study", "format": "json"}).json()

study_fields = pl.DataFrame(study_fields_raw)
print(f"Fields available for result=study: {len(study_fields)}")
print(study_fields.head(20))

### 1.2 Fetch Human RNA-seq Studies

In [ ]:
STUDIES_CACHE = DATA_DIR / "ena_human_rnaseq_studies.json"

# Fields to retrieve for each study record
STUDY_FIELDS = ",".join([
    "study_accession",
    "study_title",
    "study_type",
    "first_public",
    "last_updated",
    "center_name",
    "tax_id",
])

PAGE_SIZE = 100  # records per page (ENA Portal API maximum)


def fetch_human_rnaseq_studies(cache_path: Path = STUDIES_CACHE) -> list[dict]:
    """
    Fetch all ENA studies for Homo sapiens with library_strategy=RNA-Seq.

    Uses the ENA Portal API search endpoint with ``tax_tree(9606)`` to capture
    studies filed under any sub-taxon of human (e.g. cell lines with specific
    taxon IDs), combined with a ``library_strategy`` filter for RNA-Seq.

    Results are paginated at ``PAGE_SIZE`` records per request and cached to
    disk as a JSON file so subsequent notebook runs are instant.

    Parameters
    ----------
    cache_path : Path
        File path for the on-disk JSON cache.

    Returns
    -------
    list[dict]
        One dict per study record, with keys matching ``STUDY_FIELDS``.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_studies = []
    offset = 0

    while True:
        resp = ena_get("search", {
            "result":   "study",
            "query":    "tax_tree(9606) AND library_strategy=RNA-Seq",
            "fields":   STUDY_FIELDS,
            "limit":    PAGE_SIZE,
            "offset":   offset,
            "format":   "json",
        })

        # ENA returns an empty list (not an error) when past the last page
        batch = resp.json()
        if not batch:
            break

        all_studies.extend(batch)
        print(f"  Fetched {len(all_studies)} studies so far (offset={offset})", end="\r")
        offset += PAGE_SIZE
        time.sleep(0.3)  # additional courtesy delay during bulk pagination

    print(f"\nDone. Total studies fetched: {len(all_studies)}")
    cache_path.write_text(json.dumps(all_studies))
    print(f"Cached to: {cache_path}")
    return all_studies


studies_raw = fetch_human_rnaseq_studies()
print(f"Records in cache: {len(studies_raw)}")

### 1.3 Parse Studies into a Polars DataFrame

In [ ]:
def parse_studies(records: list[dict]) -> pl.DataFrame:
    """
    Parse raw study records from the ENA Portal API into a typed Polars DataFrame.

    Parameters
    ----------
    records : list[dict]
        Raw JSON records as returned by ``fetch_human_rnaseq_studies``.

    Returns
    -------
    pl.DataFrame
        Typed DataFrame with columns:
        study_accession (str), study_title (str), study_type (str),
        first_public (Date), last_updated (Date), center_name (str),
        tax_id (Int64).
    """
    df = pl.DataFrame(records)

    df = df.with_columns([
        # Parse ISO date strings; strict=False silently coerces blanks to null
        pl.col("first_public").str.to_date("%Y-%m-%d", strict=False),
        pl.col("last_updated").str.to_date("%Y-%m-%d", strict=False),
        # tax_id arrives as a string from the API; cast to integer
        pl.col("tax_id").cast(pl.Int64, strict=False),
    ])

    # Ensure consistent column order matching the requested fields
    return df.select([
        "study_accession",
        "study_title",
        "study_type",
        "first_public",
        "last_updated",
        "center_name",
        "tax_id",
    ])


studies = parse_studies(studies_raw)

print(f"Shape : {studies.shape}")
print(f"\nDtypes:")
print(studies.dtypes)
print(f"\nHead:")
studies.head(10)

### 1.4 Fetch Run-level Metadata for a Study of Interest

In [ ]:
# PRJEB2445: a landmark human transcriptome study (Illumina Body Map 2.0).
# It includes RNA-seq data from 16 human tissues and is widely used as a
# reference dataset, making it an ideal focus for run-level exploration.
FOCUS_STUDY = "PRJEB2445"

RUNS_CACHE = DATA_DIR / f"ena_runs_{FOCUS_STUDY}.json"

# Fields to retrieve at the run level
RUN_FIELDS = ",".join([
    "run_accession",
    "experiment_accession",
    "instrument_model",
    "library_strategy",
    "read_count",
    "base_count",
])


def fetch_study_runs(study_accession: str, cache_path: Path) -> list[dict]:
    """
    Fetch all run records belonging to a specific ENA study.

    Queries the Portal API ``search`` endpoint with ``result=read_run``
    and a study_accession filter. Paginates until no further records are
    returned, then caches the combined result as JSON.

    Parameters
    ----------
    study_accession : str
        ENA study accession (e.g. "PRJEB2445").
    cache_path : Path
        File path for the on-disk JSON cache.

    Returns
    -------
    list[dict]
        One dict per run with keys matching ``RUN_FIELDS``.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_runs = []
    offset = 0

    while True:
        resp = ena_get("search", {
            "result":  "read_run",
            "query":   f"study_accession={study_accession}",
            "fields":  RUN_FIELDS,
            "limit":   PAGE_SIZE,
            "offset":  offset,
            "format":  "json",
        })
        batch = resp.json()
        if not batch:
            break

        all_runs.extend(batch)
        print(f"  Fetched {len(all_runs)} runs (offset={offset})", end="\r")
        offset += PAGE_SIZE
        time.sleep(0.3)

    print(f"\nDone. Total runs for {study_accession}: {len(all_runs)}")
    cache_path.write_text(json.dumps(all_runs))
    print(f"Cached to: {cache_path}")
    return all_runs


runs_raw = fetch_study_runs(FOCUS_STUDY, RUNS_CACHE)

### 1.5 Parse Runs into a Polars DataFrame

In [ ]:
def parse_runs(records: list[dict]) -> pl.DataFrame:
    """
    Parse raw run records from the ENA Portal API into a typed Polars DataFrame.

    Parameters
    ----------
    records : list[dict]
        Raw JSON records as returned by ``fetch_study_runs``.

    Returns
    -------
    pl.DataFrame
        Typed DataFrame with columns:
        run_accession (str), experiment_accession (str),
        instrument_model (str), library_strategy (str),
        read_count (Int64), base_count (Int64).

    Notes
    -----
    ``read_count`` and ``base_count`` arrive as strings from the API;
    empty strings are coerced to null by ``strict=False``.
    """
    df = pl.DataFrame(records)

    df = df.with_columns([
        # Numeric columns arrive as strings; cast and treat blanks as null
        pl.col("read_count").cast(pl.Int64, strict=False),
        pl.col("base_count").cast(pl.Int64, strict=False),
    ])

    return df.select([
        "run_accession",
        "experiment_accession",
        "instrument_model",
        "library_strategy",
        "read_count",
        "base_count",
    ])


runs = parse_runs(runs_raw)

# ── Summary ───────────────────────────────────────────────────────────────────
print("=== Studies DataFrame ===")
print(f"Shape  : {studies.shape}")
print(f"Dtypes : {dict(zip(studies.columns, [str(d) for d in studies.dtypes]))}")
print()
print(studies.head(5))

print()
print(f"=== Runs DataFrame ({FOCUS_STUDY}) ===")
print(f"Shape  : {runs.shape}")
print(f"Dtypes : {dict(zip(runs.columns, [str(d) for d in runs.dtypes]))}")
print()
print(runs.head(10))